#### Reinforcement Learning Agent for Smart charging Decisions

This notebook implements an intelligent EV charging agent using **Deep Q-Networks (DQN)**.

**Problem setting:** An electric taxi driver arrives home at 2 p.m. and must leave at 4 p.m. 
During the 2-hour window (8 × 15-min slots), an automated agent decides the charging power 
every 15 minutes. The energy demand after departure is stochastic (Normal distribution). 
The agent must balance minimizing exponential charging costs against the risk of insufficient charge.

**Approach:**
- **Environment:** SimPy-based discrete-event simulation wrapped in a Gymnasium interface
- **Agent:** Deep Q-Network with experience replay and target network
- **Actions:** 4 discrete charging levels (off / low / medium / high → 0, 7.3, 14.7, 22 kW)



In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import deque, namedtuple
from itertools import count

import simpy
import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


#### Markov Decision Process Definition

#### State Space
Each state $s_t$ is a 3-dimensional continuous vector:

| Component | Description | Range |
|-----------|-------------|-------|
| SoC | Current battery state of charge (kWh) | $[0, C_{\text{max}}]$ |
| Time slot | Current 15-min period index | $\{0, 1, \ldots, 7\}$ |
| $c_t$ | Time-varying electricity cost coefficient | $[c_{\min}, c_{\max}]$ |

#### Action Space
Four discrete charging levels, mapped to continuous power values:

| Action | Label | Power (kW) | Energy per slot (kWh) |
|--------|-------|-----------|----------------------|
| 0 | Off | 0.0 | 0.0 |
| 1 | Low | 7.3 | 1.83 |
| 2 | Medium | 14.7 | 3.67 |
| 3 | High | 22.0 | 5.50 |

#### Reward Function
$$R_t = -c_t \cdot e^{p_t}$$

At episode end (after slot 7), an additional penalty applies if the accumulated charge is below the stochastic demand $D \sim \mathcal{N}(\mu=30, \sigma=5)$:

$$R_{\text{penalty}} = -\lambda \cdot \max(0, D - \text{SoC}_{\text{final}})$$

where $\lambda$ is a large penalty coefficient.

#### Transition Dynamics
$$\text{SoC}_{t+1} = \min(\text{SoC}_t + p_t \cdot \Delta t, \; C_{\text{max}})$$


In [ ]:
# ── Environment Configuration ──
ENV_CONFIG = {
    "battery_capacity_kwh": 60.0,      # Maximum battery capacity
    "initial_soc_kwh": 10.0,           # Battery level on arrival at 2 p.m.
    "n_slots": 8,                       # 2 hours / 15 min = 8 decision points
    "slot_duration_h": 0.25,            # 15 minutes in hours
    "max_power_kw": 22.0,              # Maximum charging rate
    "demand_mean_kwh": 30.0,           # μ of energy demand distribution
    "demand_std_kwh": 5.0,             # σ of energy demand distribution
    "penalty_coeff": 50.0,             # λ — penalty per kWh shortfall
    "power_levels_kw": [0.0, 7.3, 14.7, 22.0],  # Discrete action mapping
}

# Time-varying cost coefficients c_t (8 slots, 14:00–16:00)
# Source: https://hourlypricing.comed.com/live-prices/year/
# Modeled after typical afternoon electricity pricing: higher towards peak hours
ENV_CONFIG["cost_coefficients"] = np.array([
    3.2, 3.4, 3.6, 3.8, 4.0, 4.1, 4.2, 4.2
])

print("Energy per slot (kWh) per action:")
for i, p in enumerate(ENV_CONFIG["power_levels_kw"]):
    print(f"  Action {i} ({p:5.1f} kW) → {p * ENV_CONFIG['slot_duration_h']:.2f} kWh")
print(f"\nMax chargeable in 2h: {ENV_CONFIG['max_power_kw'] * ENV_CONFIG['n_slots'] * ENV_CONFIG['slot_duration_h']:.1f} kWh")
print(f"Demand distribution: N({ENV_CONFIG['demand_mean_kwh']}, {ENV_CONFIG['demand_std_kwh']}²)")


Energy per slot (kWh) per action:
  Action 0 (  0.0 kW) → 0.00 kWh
  Action 1 (  7.3 kW) → 1.82 kWh
  Action 2 ( 14.7 kW) → 3.67 kWh
  Action 3 ( 22.0 kW) → 5.50 kWh

Max chargeable in 2h: 44.0 kWh
Demand distribution: N(30.0, 5.0²)


In [8]:
class EVChargingEnv(gym.Env):
    """
    Electric Vehicle Smart Charging Environment.
    
    Uses SimPy internally for discrete-event simulation of the 2-hour charging
    window. Exposes a standard Gymnasium interface for RL agent interaction.
    """
    
    metadata = {"render_modes": ["human"]}
    
    def __init__(self, config=None):
        super().__init__()
        self.cfg = config or ENV_CONFIG
        
        # Action space: 4 discrete charging levels
        self.action_space = spaces.Discrete(len(self.cfg["power_levels_kw"]))
        
        # Observation space: [SoC, time_slot (normalized), cost_coefficient]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([self.cfg["battery_capacity_kwh"], 1.0, 1.0], dtype=np.float32),
        )
        
        # Internal state
        self.soc = 0.0
        self.current_slot = 0
        self.episode_cost = 0.0
        self.slot_history = []
        
        # SimPy placeholders
        self._simpy_env = None
        self._action_event = None
        self._step_result = None
        
    def _get_obs(self):
        """Construct observation vector."""
        normalized_slot = self.current_slot / (self.cfg["n_slots"] - 1)
        c_t = self.cfg["cost_coefficients"][min(self.current_slot, self.cfg["n_slots"] - 1)]
        return np.array([self.soc, normalized_slot, c_t], dtype=np.float32)
    
    def _charging_process(self, env):
        """SimPy process: runs through all 8 charging slots."""
        for slot in range(self.cfg["n_slots"]):
            self.current_slot = slot
            
            # Wait for agent's action
            self._action_event = env.event()
            yield self._action_event
            action = self._action_event.value
            
            # Apply charging
            power_kw = self.cfg["power_levels_kw"][action]
            energy_kwh = power_kw * self.cfg["slot_duration_h"]
            old_soc = self.soc
            self.soc = min(self.soc + energy_kwh, self.cfg["battery_capacity_kwh"])
            actual_energy = self.soc - old_soc
            
            # Compute cost: c_t * e^p
            c_t = self.cfg["cost_coefficients"][slot]
            slot_cost = c_t * np.exp(power_kw)
            self.episode_cost += slot_cost
            
            # Reward is negative cost
            reward = -slot_cost
            
            # Log
            self.slot_history.append({
                "slot": slot,
                "action": action,
                "power_kw": power_kw,
                "energy_kwh": actual_energy,
                "soc_after": self.soc,
                "cost": slot_cost,
                "c_t": c_t,
            })
            
            terminated = (slot == self.cfg["n_slots"] - 1)
            
            # At episode end: realize demand and apply penalty
            penalty = 0.0
            demand = None
            if terminated:
                demand = np.random.normal(
                    self.cfg["demand_mean_kwh"], self.cfg["demand_std_kwh"]
                )
                demand = max(0, demand)  # Demand cannot be negative
                shortfall = max(0, demand - self.soc)
                if shortfall > 0:
                    penalty = self.cfg["penalty_coeff"] * shortfall
                    reward -= penalty
            
            self._step_result = (
                self._get_obs(), reward, terminated, False,
                {"cost": slot_cost, "penalty": penalty, "demand": demand, "soc": self.soc}
            )
            
            # Advance SimPy by 15 minutes
            yield env.timeout(15)
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.soc = self.cfg["initial_soc_kwh"]
        self.current_slot = 0
        self.episode_cost = 0.0
        self.slot_history = []
        
        # Initialize SimPy environment and start charging process
        self._simpy_env = simpy.Environment()
        self._simpy_env.process(self._charging_process(self._simpy_env))
        # Run until the process yields the first action_event
        self._simpy_env.step()
        
        return self._get_obs(), {}
    
    def step(self, action):
        """Execute one charging decision."""
        # Trigger the action event in SimPy
        self._action_event.succeed(value=action)
        
        # Advance SimPy simulation (processes the action + timeout)
        try:
            self._simpy_env.step()  # Process action
            self._simpy_env.step()  # Process timeout → next slot or end
        except StopIteration:
            pass  # End of simulation
        
        return self._step_result


# Quick sanity check
env = EVChargingEnv(ENV_CONFIG)
obs, _ = env.reset()
print(f"Initial observation: SoC={obs[0]:.1f} kWh, slot={obs[1]:.2f}, c_t={obs[2]:.2f}")

total_reward = 0
for step in range(8):
    obs, reward, terminated, truncated, info = env.step(3)  # Always charge at max
    total_reward += reward
    print(f"  Slot {step}: SoC={info['soc']:.1f} kWh, cost={info['cost']:.2f}, reward={reward:.2f}", end="")
    if terminated:
        print(f"  | Demand={info['demand']:.1f}, penalty={info['penalty']:.2f}")
    else:
        print()
print(f"Total reward (always-max policy): {total_reward:.2f}")


Initial observation: SoC=10.0 kWh, slot=0.00, c_t=0.04
  Slot 0: SoC=15.5 kWh, cost=143396513.85, reward=-143396513.85
  Slot 1: SoC=21.0 kWh, cost=179245642.31, reward=-179245642.31
  Slot 2: SoC=26.5 kWh, cost=215094770.77, reward=-215094770.77
  Slot 3: SoC=32.0 kWh, cost=250943899.23, reward=-250943899.23
  Slot 4: SoC=37.5 kWh, cost=286793027.69, reward=-286793027.69
  Slot 5: SoC=43.0 kWh, cost=322642156.15, reward=-322642156.15
  Slot 6: SoC=48.5 kWh, cost=358491284.61, reward=-358491284.61
  Slot 7: SoC=54.0 kWh, cost=394340413.07, reward=-394340413.07  | Demand=29.3, penalty=0.00
Total reward (always-max policy): -2150947707.68
